# Regge-sector term-by-term comparison

This notebook prepares the numerical data entering $B_\alpha$, $C_\alpha$, $\mathfrak h_{\alpha\beta}$, and $\mathcal I_{\rm bdry}$, then compares the $S'_0$ Gaussian expansion with the Regge linear and quadratic terms at $t_i=0$.


In [1]:
using Pkg
root_path = joinpath(pwd(), "..")
Pkg.activate(root_path)
Pkg.resolve()
Pkg.instantiate()

using EffectiveSpinfoam
using LorentzianSimplexSolver
using SymEngine
using LinearAlgebra

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam`
     Project No packages added to or removed from `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/Project.toml`
    Manifest No packages added to or removed from `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/Manifest.toml`


## 1. Input data

The example below uses the same triangulation as the main numerical workflow. The default perturbation list contains all bulk edges and one selected boundary edge.


In [2]:
simplices = [[1,2,3,4,6], [1,2,3,5,6], [1,2,4,5,6],[1,2,3,4,7], [1,2,3,5,7], [1,2,4,5,7]]

coords_lines = [
    "0, 0, 0, 0",
    "-0.068000000000000005, -0.21988127663727278, -0.5316227766016838, -1.3316227766016839",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "-2.4696884592430974, -3.893218630529324, -1.3565336794679874, -1.9090667752920147",
]

ScalarT = Float64
# ScalarT = BigFloat
tol = LorentzianSimplexSolver.configure_precision!(ScalarT; precision=80)

gamma_value = ScalarT(1.0);
vertex_coords = build_vertex_coordinates(simplices, coords_lines, ScalarT);

## 2. Solver and effective-stage data

`build_solver_data` constructs the matched critical configuration and the symbolic Hessian. `build_effective_data` adds the length-variation data needed for the Regge comparison.

In [3]:
solver = build_solver_data(simplices, vertex_coords; bulk_sum_form=false);
data = build_effective_data(solver, simplices, vertex_coords);

geom = data.geom
regge = data.regge
log_spinfoam = data.log_spinfoam;

gamma_symbol = log_spinfoam.gamma_symbol
sd = log_spinfoam.solve_data
phase_soln = log_spinfoam.phase_solution

g_vars = geom.varias[:g_var]
z_vars = geom.varias[:z_var]
X_vars = vcat(g_vars, z_vars)
nX = length(X_vars);

nh = length(geom.connectivity[1]["OrderBulkFaces"])
nb = length(geom.connectivity[1]["OrderBDryFaces"]);

## 3. Boundary Taylor terms

`compute_boundary_taylor_terms` assembles the Gaussian sources and the boundary Taylor pieces used in the Regge-sector checks. Internally it evaluates the shared Hessian/area-derivative blocks, solves the linearized boundary response for $D_\ell\xi$, and returns only the quantities needed below.


In [4]:
boundary_terms = compute_boundary_taylor_terms(data, gamma_value; tol=tol);

B = boundary_terms.B
C = boundary_terms.C
h_frak = boundary_terms.h_frak
inv_h_frak = boundary_terms.inv_h_frak

dkb_dl = boundary_terms.dkb_dl
d2kb_dldl = boundary_terms.d2kb_dldl
dηdl = boundary_terms.dηdl

I_bdy_linear = boundary_terms.I_bdy_linear
I_bdy_quadratic = boundary_terms.I_bdy_quadratic;

## 4. Regge terms

The Regge linear coefficient is

$$
iS_{\rm Regge}^{(1)}
=
\frac{i\gamma}{2}\,(D_\ell k_b)^T\Theta_b,
$$

In [5]:
dihedral_angles = regge.dihedral_angles;

# i S_Regge^(1)
iSRegge_linear = im / 2 * gamma_value * transpose(dkb_dl) * dihedral_angles;

The Regge bulk quadratic coefficients are
$$
iS_{{\rm Regge},{\rm bulk}}^{(2)}
=
\frac{i\gamma}{4}\,(D_\ell\eta_h)^T D_\ell\varepsilon_h,
$$

In [6]:
iSRegge_bulk_quadratic = im * gamma_value / 4 * transpose(dηdl) * data.DεDl;

The Regge boundary quadratic coefficients are
$$
iS_{{\rm Regge},{\rm bdry}}^{(2)}
=
\frac{i\gamma}{4}
\left[
(D_\ell k_b)^T D_\ell\Theta_b
+
\sum_b \Theta_b\,D_\ell^2 k_b
\right],
$$

In [7]:
iSRegge_boundary_quadratic = im * gamma_value / 4 * (
    transpose(dkb_dl) * data.DΘDl +
    sum(dihedral_angles[i] * d2kb_dldl[i, :, :] for i in 1:nb)
);

The Regge quadratic term is
$$
iS_{\rm Regge}^{(2)}
=
iS_{{\rm Regge},{\rm bulk}}^{(2)}
+
iS_{{\rm Regge},{\rm bdry}}^{(2)}.
$$

In [8]:
iSRegge_quadratic = iSRegge_bulk_quadratic + iSRegge_boundary_quadratic;

The term-by-term Regge-sector comparison is

$$
-\frac12 B_\alpha(\mathfrak h^{-1})^{\alpha\beta}(B_\beta+C_\beta)
=
iS_{{\rm Regge},{\rm bulk}}^{(2)},
$$


In [9]:
max_abs_entry(value::Number) = abs(value)
max_abs_entry(value) = maximum(abs.(value))

function report_comparison(name, lhs, rhs)
    diff = lhs - rhs
    println(name)
    println("lhs:")
    show(stdout, "text/plain", lhs)
    println()
    println("rhs:")
    show(stdout, "text/plain", rhs)
    println()
    println("max |lhs - rhs| = ", max_abs_entry(diff))
    return diff
end;

In [10]:
bulk_quadratic_lhs = -1/2 * transpose(B) * inv_h_frak * (B + C);
bulk_quadratic_rhs = iSRegge_bulk_quadratic;

bulk_quadratic_difference = report_comparison(
    "Bulk quadratic term",
    bulk_quadratic_lhs,
    bulk_quadratic_rhs,
);
println()

Bulk quadratic term
lhs:
2×2 Matrix{ComplexF64}:
  2.62093e-11-134.089im  -8.25193e-12+152.539im
 -7.68472e-12+28.0163im   2.41375e-12-55.5126im
rhs:
2×2 Matrix{ComplexF64}:
 -0.0-134.089im   0.0+152.539im
  0.0+28.0163im  -0.0-55.5126im
max |lhs - rhs| = 2.795374756939448e-11



$$
\mathcal I_{\rm bdry}^{(1)}
=
iS_{\rm Regge}^{(1)},
$$

In [11]:
boundary_linear_lhs = transpose(I_bdy_linear);
boundary_linear_rhs = iSRegge_linear;

boundary_linear_difference = report_comparison(
    "Boundary linear term",
    boundary_linear_lhs,
    boundary_linear_rhs,
);
println()

Boundary linear term
lhs:
2-element Vector{ComplexF64}:
   3.484777681524342e-11 + 2.118424588669903e-11im
 -1.1054904869371534e-11 - 2.478122441376952im
rhs:
2-element Vector{Any}:
 0.0 + 0.0im
 0.0 - 2.478122441362764im
max |lhs - rhs| = 4.0781611331004826e-11



$$
-\frac12 C_\alpha(\mathfrak h^{-1})^{\alpha\beta}(B_\beta+C_\beta)
+
\mathcal I_{\rm bdry}^{(2)}
=
iS_{{\rm Regge},{\rm bdry}}^{(2)}.
$$

In [12]:
boundary_quadratic_lhs = -1/2 * transpose(C) * inv_h_frak * (B + C) + I_bdy_quadratic;
boundary_quadratic_rhs = iSRegge_boundary_quadratic;

boundary_quadratic_difference = report_comparison(
    "Boundary quadratic term",
    boundary_quadratic_lhs,
    boundary_quadratic_rhs,
);

Boundary quadratic term
lhs:
2×2 Matrix{ComplexF64}:
  1.65395e-22-9.86814e-23im  -3.35962e-13+4.01588e-13im
 -9.13069e-13+124.523im      -1.33227e-15-62.2403im
rhs:
2×2 Matrix{ComplexF64}:
 0.0+0.0im       0.0+0.0im
 0.0+124.523im  -0.0-62.2403im
max |lhs - rhs| = 4.777643250395378e-12


Putting the pieces together gives the $t_i=0$ Regge-sector identity

$$
S'_{0,{\rm eff}}(\ell)-\bar S'_0
=
iS_{\rm Regge}^{(1)}
+
iS_{\rm Regge}^{(2)}
+O(\delta\ell^3).
$$

The following comparison is therefore displayed as a linear-vector check and a quadratic-matrix check.


In [13]:
full_linear_lhs = transpose(I_bdy_linear);
full_linear_rhs = iSRegge_linear;

full_quadratic_lhs = -1/2 * transpose(B + C) * inv_h_frak * (B + C) + I_bdy_quadratic;
full_quadratic_rhs = iSRegge_quadratic;

full_linear_difference = report_comparison(
    "Full linear term",
    full_linear_lhs,
    full_linear_rhs,
);
println()

full_quadratic_difference = report_comparison(
    "Full quadratic term",
    full_quadratic_lhs,
    full_quadratic_rhs,
);

Full linear term
lhs:
2-element Vector{ComplexF64}:
   3.484777681524342e-11 + 2.118424588669903e-11im
 -1.1054904869371534e-11 - 2.478122441376952im
rhs:
2-element Vector{Any}:
 0.0 + 0.0im
 0.0 - 2.478122441362764im
max |lhs - rhs| = 4.0781611331004826e-11

Full quadratic term
lhs:
2×2 Matrix{ComplexF64}:
  2.62186e-11-134.089im  -8.58961e-12+152.539im
 -8.58566e-12+152.539im   2.40696e-12-117.753im
rhs:
2×2 Matrix{ComplexF64}:
 0.0-134.089im   0.0+152.539im
 0.0+152.539im  -0.0-117.753im
max |lhs - rhs| = 2.7952533173395897e-11
